# 12 . When should we trade? Time-of-day & day-of-week structure

Hypothesis: some hours/days are **choppy** — an EMA-based approach gives fewer clean moves and loses
money — so we should block them, and re-run the analysis periodically to keep the list current.

The trap: with 24 hours (or 7 days) to choose from, an in-sample ranking is **never** flat. You will
always find "good" and "bad" buckets. The only question that matters is whether the **same** buckets
stay good in the *next* period. So this notebook is built around three steps:

1. **Measure** structure in a direction-agnostic way (so a bull/bear drift can't fake a result):
   - *efficiency ratio* — is this hour a clean one-way move, or chop?
   - *EMA follow-through* — after a price↔EMA cross, does price travel in the signal's direction?
2. **Test persistence** — rank-correlate consecutive periods, and walk-forward the blocking rule.
3. **Only then** apply it to the strategy, and check it holds in a regime it wasn't chosen on.

> Set `ROLL_FREQ` in section 1 to `'M'`, `'Q'`, or `'2W'` to re-run the whole analysis at whatever
> cadence you want — that is the "rolling analysis" knob.

## 0 . Setup

In [ ]:
import sys, os, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant
import numpy as np, pandas as pd
from quant.data import get_ohlcv
from quant.data.timeframe import resample_ohlcv
from quant.analytics import (efficiency_by_block, ema_followthrough, bucket_by_period,
                             persistence, walkforward_block, trade_buckets, SESSIONS)
from quant import scenarios as sc
pd.set_option('display.width', 260); pd.set_option('display.max_rows', 60)
WD = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

## 1 . Data + the rolling cadence

In [ ]:
SYMBOL, TF, SOURCE = 'BTCUSDT', '1m', 'binance'
START, END = '2025-01-01', '2026-07-29'
ROLL_FREQ  = 'Q'      # <- the rolling window: 'M' monthly | 'Q' quarterly | '2W' fortnightly
EMA_PERIOD = 50       # EMA whose crosses we measure
FWD_BARS   = 30       # how far ahead to measure follow-through (bars of the TF)

df = get_ohlcv(SYMBOL, TF, start=START, end=END, source=SOURCE, tz='UTC')
# regime split — used later to check a filter on data it was NOT chosen from
SPLIT = '2025-07-29'
BULL, BEAR = df[df.t < SPLIT].reset_index(drop=True), df[df.t >= SPLIT].reset_index(drop=True)
for nm, d in [('FULL', df), ('BULL', BULL), ('BEAR', BEAR)]:
    print(f'{nm:5s} {len(d):>8,} bars  {d.t.min():%Y-%m-%d} -> {d.t.max():%Y-%m-%d}  '
          f'BTC {d.close.iloc[-1]/d.close.iloc[0]-1:+.1%}')

## 2 . Chop map — efficiency ratio by hour
`1.0` = a clean one-way move within the hour, `~0` = pure chop. Direction-agnostic.

In [ ]:
er = efficiency_by_block(df, freq='h')
t = er.groupby('hour').er.agg(['mean','median','count']).rename(columns={'mean':'efficiency'})
t['session'] = [SESSIONS[h] for h in t.index]
print(t.round(4).to_string())
print('\nspread across hours: %.4f -> %.4f  (%.0f%% relative)'
      % (t.efficiency.min(), t.efficiency.max(),
         (t.efficiency.max()/t.efficiency.min()-1)*100))

## 3 . EMA follow-through by hour, and across timeframes
If the hour-of-day effect were real, the best/worst hours should broadly agree between 1m, 5m and 15m.

In [ ]:
ft = ema_followthrough(df, period=EMA_PERIOD, fwd=FWD_BARS)
h = ft.groupby('hour').ft.agg(['mean','count']).rename(columns={'mean':'followthrough_ATR'})
h['session'] = [SESSIONS[i] for i in h.index]
print(h.round(4).to_string())

print('\nagreement across timeframes:')
for tf, fw in [('1min', FWD_BARS), ('5min', 24), ('15min', 16)]:
    d = df if tf == '1min' else resample_ohlcv(df, tf)
    m = ema_followthrough(d, period=EMA_PERIOD, fwd=fw).groupby('hour').ft.mean()
    print(f'  {tf:6s} best {list(m.nlargest(4).index)}   worst {list(m.nsmallest(4).index)}')

## 4 . ⚠ THE DECIDING TEST — does the ranking persist?
Rank-correlate the hourly means between consecutive `ROLL_FREQ` periods. **Mean rho near 0 (or
negative) means the ranking reshuffles every period and any hour filter is curve-fitting.**

In [ ]:
tab_h = bucket_by_period(ft, bucket='hour', freq=ROLL_FREQ)
print('per-hour follow-through, period by period:')
print(tab_h.round(3).to_string())
p_h = persistence(tab_h)
print('\nconsecutive-period rank correlation:')
print(p_h.to_string(index=False))
print('  MEAN rho = %+.3f' % p_h.attrs['mean_rho'])

## 5 . Walk-forward the blocking rule
Block the worst N hours found in each period, then score the **next** period. For blocking to be
worth doing, `improved` must be true much more often than half the time.

In [ ]:
wf = walkforward_block(ft, tab_h, bucket='hour', n_block=6, freq=ROLL_FREQ)
print(wf.to_string(index=False))
print('\n  blocking helped in %d of %d periods (coin-flip = %.1f)'
      % (wf.improved.sum(), len(wf), len(wf)/2))

## 6 . Same three tests for DAY OF WEEK
Only 7 buckets instead of 24, so each holds far more data — more statistical power.

In [ ]:
tab_w = bucket_by_period(ft, bucket='weekday', freq=ROLL_FREQ)
tab_w.index = [WD[i] for i in tab_w.index]
print(tab_w.round(3).to_string())
p_w = persistence(tab_w)
print('\nconsecutive-period rank correlation:')
print(p_w.to_string(index=False))
print('  MEAN rho = %+.3f   (compare with the hourly figure in section 4)' % p_w.attrs['mean_rho'])
print('\nhow often is each day positive across the %d periods?' % tab_w.shape[1])
print((tab_w > 0).sum(axis=1).to_string())

## 7 . Now the strategy itself — expectancy by bucket, in two regimes
A bucket only interests us if it is positive in a regime it was **not** picked from.

In [ ]:
SCEN, STOP, TP = 'support_break_short@v1.2', 'leg_high@v1', 'trailing@v1'
P = dict(entry_mode='break_close', break_mode='close_below', prev_low_n=20, confirm_red=2,
         grad_period=200, grad_smooth=9, grad_lookback=5, grad_max=0.0,
         stop_buffer_pct=0.05, stop_pct=0.6, trail_pct=1.0)
CFG = dict(initial_cash=10_000, spread=6.0,           # ~0.7bp — costs matter, see notebook 11 notes
           sizing_mode='risk_pct_equity', sizing_value=1.0,
           allow_leverage=True, max_notional_pct=5000)  # BOTH needed or risk sizing silently caps

def buckets(d, bucket):
    return trade_buckets(sc.run_pieces(d, SCEN, STOP, TP, params=P, cfg=CFG).trades, bucket=bucket)

for bucket in ['hour', 'weekday']:
    b, u = buckets(BEAR, bucket), buckets(BULL, bucket)
    cmp = pd.DataFrame({'bear_R': b.expectancy_R, 'bear_n': b.trades,
                        'bull_R': u.expectancy_R, 'bull_n': u.trades})
    if bucket == 'weekday':
        cmp.index = [WD[i] for i in cmp.index]
    cmp['both_pos'] = (cmp.bear_R > 0) & (cmp.bull_R > 0)
    print(f'--- by {bucket} ---'); print(cmp.round(3).to_string())
    print(f'  positive in BOTH regimes: {list(cmp[cmp.both_pos].index)}'
          f'   (by chance you would expect ~{len(cmp)//4})\n')

## 8 . Apply a weekday filter and check the unseen regime
`weekdays=(0..6)` is a built-in strategy filter, so this is one param. **Caveat:** any day-set chosen
by looking at sections 6–7 is in-sample; treat `both_positive` as encouraging, not as proof.

In [ ]:
def expR(d, **over):
    r = sc.run_pieces(d, SCEN, STOP, TP, params={**P, **over}, cfg=CFG)
    t = r.trades
    if not len(t):
        return None, 0
    risk = (t.stop_price - t.entry_price).abs() * t.qty
    return round((t.pnl / risk).mean(), 4), len(t)

rows = []
for tag, wds in [('all days', None), ('Sun only', (6,)), ('exclude Sat', (0,1,2,3,4,6)),
                 ('Mon+Thu+Sun', (0,3,6)), ('exclude Wed+Sat', (0,1,3,4,6))]:
    over = {} if wds is None else {'weekdays': wds}
    rb, nb = expR(BEAR, **over); ru, nu = expR(BULL, **over)
    rows.append(dict(filter=tag, bear_expR=rb, bear_n=nb, bull_expR=ru, bull_n=nu,
                     both_positive=(rb or 0) > 0 and (ru or 0) > 0))
pd.DataFrame(rows)

## 9 . How to read this
**The bar to clear before blocking anything:**
1. mean rho in section 4/6 clearly **> 0** (say > 0.3), and
2. walk-forward `improved` in section 5 clearly better than half, and
3. the bucket still positive in the regime it was **not** chosen from (section 7/8).

If a bucket fails these, it is noise — blocking it will look great in the backtest and do nothing
(or harm) live. Re-run this notebook at your chosen `ROLL_FREQ` before each re-optimisation, and
watch whether the same buckets keep showing up; a genuine effect drifts slowly, noise reshuffles.

> Sample-size discipline: an hourly bucket over one quarter holds only a few hundred signals and a
> few dozen trades. Prefer weekday (7 buckets) or session groupings (4–6) over raw hours (24) unless
> you have years of data.